In [1]:
import os

In [2]:
%pwd

'd:\\Bappy\\YouTube\\Kidney-Disease-Classification-Deep-Learning-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Bappy\\YouTube\\Kidney-Disease-Classification-Deep-Learning-Project'

In [1]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [2]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories



In [3]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [8]:
import os
import zipfile
import kagglehub
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

import mlflow
import os
import shutil

# Define the tracking path on your NAS
tracking_uri = "file:///W:/ML-DL/Repositories/kidney-disease-classification-mlflow/mlruns"
mlflow.set_tracking_uri(tracking_uri)

# Create or set the experiment name
mlflow.set_experiment("Kidney_Disease_Classification")

print(f"MLflow is now tracking to: {mlflow.get_tracking_uri()}")

# import os

# # Set the custom directory to your artifacts folder
# # os.environ["KAGGLEHUB_CACHE"] = "artifacts/data_ingestion"

# import kagglehub

# # Now, any download will go into artifacts/data_ingestion
# dataset_path = kagglehub.dataset_download("username/dataset-name")

# print(f"Data is now stored in: {dataset_path}")



MLflow is now tracking to: file:///W:/ML-DL/Repositories/kidney-disease-classification-mlflow/mlruns


In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            os.environ["KAGGLEHUB_CACHE"] = self.config.root_dir
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(self.config.unzip_dir, exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            # kagglehub.dataset_download(dataset_url, zip_download_dir)
            cache_path = kagglehub.dataset_download("nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone")

            os.makedirs("artifacts/data_ingestion/test", exist_ok=True)
            
            for item in os.listdir(cache_path):
                s = os.path.join(cache_path, item)
                d = os.path.join("artifacts/data_ingestion", item)
                if os.path.isdir(s):
                    shutil.copytree(s, d, dirs_exist_ok=True)
                else:
                    shutil.copy2(s, d)

            logger.info(f"Data successfully moved to: artifacts/data_ingestion")
            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)



In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    # data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-16 11:52:23,246: INFO: common: yaml file: W:\ML-DL\Repositories\kidney-disease-classification-mlflow\config\config.yaml loaded successfully]
[2026-03-16 11:52:23,276: INFO: common: yaml file: W:\ML-DL\Repositories\kidney-disease-classification-mlflow\params.yaml loaded successfully]
[2026-03-16 11:52:23,326: INFO: common: created directory at: artifacts]
[2026-03-16 11:52:23,403: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-16 11:52:23,442: INFO: 313125881: Downloading data from nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone into file artifacts/data_ingestion/data.zip]
[2026-03-16 13:05:02,326: INFO: 313125881: Data successfully moved to: artifacts/data_ingestion]
[2026-03-16 13:05:02,341: INFO: 313125881: Downloaded data from nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone into file artifacts/data_ingestion/data.zip]
